# Chạy pipeline — bản notebook của `main.py`

Dùng notebook này khi muốn **xem từng bước và biểu đồ ngay tại chỗ**.
Khi chỉ cần kết quả cuối, chạy `python main.py --member <ten>` nhanh hơn.

Hai cách cho ra cùng một kết quả, vì đều gọi `HousePricePipeline`.

In [1]:
# Cho phép notebook import được package src/
import sys
sys.path.append('..')

# Tự nạp lại code khi file trong src/ thay đổi.
# Không có hai dòng này thì bạn sửa Clean.py mà notebook vẫn chạy bản cũ.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)

## Cách 1: chạy một lệnh, giống `main.py`

In [2]:
from src.pipeline.Pipeline import HousePricePipeline

out = HousePricePipeline(member='team').run(['lasso', 'lightgbm'])
print('File nộp:', out)

Bỏ qua model chưa khả dụng (thiếu thư viện?): ['lightgbm']


Dữ liệu sau xử lý: train (1458, 269), test (1459, 269)


lasso_{'rmse_oof': 0.11000236101985193, 'rmse_cv_mean': 0.10978424464255228, 'rmse_cv_std': 0.00685118391129233}



model  rmse_oof  rmse_cv_mean  rmse_cv_std
lasso      0.11       0.10978      0.00685


{'submission_rows': 1459, 'price_min': 40807.11061861559, 'price_max': 1037067.8239087033}


File nộp: C:\Users\Hung\OneDrive\ドキュメント\May_hoc\RepoNhom\Challege 1\2_HousePrices\SUBMIT-summarize_report\submission.csv


## Cách 2: chạy từng bước để xem chi tiết

Phần dưới làm đúng những gì `run()` làm, nhưng tách ra để xem được số liệu
ở giữa. Dùng khi cần gỡ lỗi hoặc giải thích trong báo cáo.

### Bước 1: Clean → Feature → Preprocess

In [3]:
from src.pipeline.Pipeline import HousePricePipeline, MODELS

pipe = HousePricePipeline(member='team')
X, y, X_test, test_ids = pipe.prepare()

print('X:', X.shape, '| X_test:', X_test.shape)
print('Model khả dụng:', list(MODELS))
print('y (log) min/max:', round(y.min(), 3), round(y.max(), 3))

X: (1458, 269) | X_test: (1459, 269)
Model khả dụng: ['lasso', 'elasticnet', 'kernelridge', 'randomforest']
y (log) min/max: 10.46 13.534


### Bước 2: cross-validation từng model

`oof_predict` chia 5 fold. Mỗi dòng train được dự đoán bởi model chưa từng
thấy dòng đó, nên con số này đáng tin hơn một lần chia train/test.

In [4]:
from src.evaluation.Evaluation import Evaluator

results = {}
for name in MODELS:
    results[name] = Evaluator.oof_predict(
        lambda n=name: MODELS[n](pipe.cfg), X, y, X_test,
        n_folds=pipe.cfg['split']['n_folds'], seed=pipe.cfg['project']['seed'])
    r = results[name]
    print(f"{name:10s} rmse_oof={r['rmse_oof']:.5f}  "
          f"cv={r['rmse_mean']:.5f} ± {r['rmse_std']:.5f}")

lasso      rmse_oof=0.11000  cv=0.10978 ± 0.00685


elasticnet rmse_oof=0.11100  cv=0.11077 ± 0.00702


kernelridge rmse_oof=0.11052  cv=0.11025 ± 0.00767


randomforest rmse_oof=0.13324  cv=0.13289 ± 0.00964


In [5]:
Evaluator.compare(results)

,model,rmse_oof,rmse_cv_mean,rmse_cv_std
0,lasso,0.11000,0.10978,0.00685
1,kernelridge,0.11052,0.11025,0.00767
2,elasticnet,0.11100,0.11077,0.00702
3,randomforest,0.13324,0.13289,0.00964


### Bước 3: so sánh các cách kết hợp model

Bảng này là nội dung chính cho phần ensemble của báo cáo.

In [6]:
from src.ensemble.Ensemble import compare_methods

oof = {k: v['oof'] for k, v in results.items()}
compare_methods(oof, y)

,method,rmse_oof,weights
0,stacking,0.10865,"{'lasso': 0.288, 'elasticnet': 0.263, 'kernelr..."
1,blending,0.10894,"{'lasso': 0.295, 'elasticnet': 0.291, 'kernelr..."
2,voting,0.10947,"{'lasso': 0.25, 'elasticnet': 0.25, 'kernelrid..."
3,(đơn lẻ) lasso,0.11000,{}
4,(đơn lẻ) kernelridge,0.11052,{}
5,(đơn lẻ) elasticnet,0.11100,{}
6,(đơn lẻ) randomforest,0.13324,{}


### Bước 4: biểu đồ chẩn đoán

Phần dư nên rải đều quanh 0. Nếu thấy hình phễu (càng về giá cao càng lệch)
thì model còn thiếu đặc trưng cho nhà đắt.

In [7]:
from src.utils.data_loader import get_path

best = min(results, key=lambda k: results[k]['rmse_oof'])
print('Model tốt nhất:', best)
Evaluator.plot_pred_vs_actual(y, oof[best], fig_dir=get_path(pipe.cfg, 'figures'))
Evaluator.plot_residuals(y, oof[best], fig_dir=get_path(pipe.cfg, 'figures'))

Model tốt nhất: lasso


### Bước 5: đặc trưng nào quan trọng

In [8]:
model = MODELS[best](pipe.cfg).fit(X, y)
imp = model.feature_importance(X.columns)
Evaluator.plot_feature_importance(imp, k=25, fig_dir=get_path(pipe.cfg, 'figures'))
imp.head(20)

MSZoning_C (all)         0.295423
Neighborhood_Crawfor     0.093703
TotRmsAbvGrd             0.079267
Neighborhood_StoneBr     0.075544
HouseAge                 0.074076
TotalFinSF               0.067845
SaleType_New             0.058462
OverallQual              0.056508
Exterior1st_BrkFace      0.054521
LivAreaPerRoom           0.052451
Neighborhood_BrkSide     0.051917
Neighborhood_NridgHt     0.048671
OverallGrade             0.042455
HasPorch                 0.042374
KitchenQual              0.042196
Condition1_Norm          0.039983
TotalBsmtSF              0.038319
Neighborhood_NoRidge     0.037707
Functional               0.036283
SaleCondition_Abnorml    0.035622
dtype: float64

## Sau khi chạy

- [ ] Nộp `SUBMIT-summarize_report/submission.csv` lên Kaggle
- [ ] Ghi một dòng vào `reports/results_summary.md`: ngày, người, model, RMSE CV, điểm Kaggle
- [ ] Đối chiếu: điểm Kaggle lệch dưới 0.01 so với RMSE CV là bình thường.
      Lệch nhiều hơn thì nghi có rò rỉ dữ liệu, phải tìm trước khi làm tiếp.